# Met Office Global 10 km Deterministic NWP – AWS ASDI Demo

This notebook demonstrates how to use the `site_archive_aws.MOGlobal10km` data accessor
to read Met Office Global 10 km NWP data directly from AWS S3 and create visualisations.

## Dataset
The Met Office Global Atmospheric High Resolution Model produces operational
deterministic forecasts at ~10 km horizontal grid spacing, updated every 6 hours.
Data are publicly available via the [AWS Sustainable Data Initiative](https://registry.opendata.aws/uk-met-office/).

## Requirements
```
pip install pyearthtools-archive-aws
```


In [8]:
# Standard imports
import warnings
import datetime

In [2]:
import numpy as np
import xarray as xr


In [3]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [4]:
# PyEarthTools AWS accessor
import site_archive_aws
from site_archive_aws import MOGlobal10km

print(f"site_archive_aws version: {site_archive_aws.__version__}")

ROOT_DIRECTORIES: {'MOGlobal10km': 's3://met-office-atmospheric-model-data/global-deterministic/', 'MOUKV_AWS': 's3://met-office-atmospheric-model-data/uk-deterministic/', 'MOGREPSGlobal': 's3://met-office-ensemble-model-data/global-ensemble/', 'MOGREPSUK': 's3://met-office-ensemble-model-data/uk-ensemble/'}
site_archive_aws version: 999


## 1. Configure the Accessor

We use `anon=True` for anonymous (unauthenticated) access to the public Met Office bucket.
Remove this flag (or set `anon=False`) if you have AWS credentials configured.

In [9]:
# Choose a date and time that has data available.
# The Global 10km model runs at 00Z, 06Z, 12Z, 18Z.
QUERY_TIME = datetime.datetime(2025,8,14,12,0)

In [13]:


# Create the accessor for 2-m temperature
# anon=True → no AWS credentials required for public data
accessor_2t = MOGlobal10km("2t", anon=True)
print(accessor_2t)

MOGlobal10km
	Description                    Met Office Global 10 km Deterministic NWP (AWS ASDI)
		 range                          '2019–present'
		 Documentation                  'https://registry.opendata.aws/uk-met-office/'


	Initialisation                 
		 anon                           True
		 forecast_hour                  None
		 level_value                    None
		 variables                      '2t'
	Transforms                     
		 StandardCoordinateNames        {'latitude': "['lat', 'Latitude', 'yt_ocean', 'yt']", 'longitude': "['lon', 'Longitude', 'xt_ocean', 'xt']", 'replacement_dictionary': 'None', 'time': "['Time']"}
		 Trim                           {'__args': '()', 'variables': "['2t']"}


## 2. Load the Data

Indexing the accessor with a datetime string loads data lazily from S3.

In [11]:
# Load 2-m temperature for the chosen time
ds_2t = accessor_2t[QUERY_TIME]
print(ds_2t)
print("\nVariables:", list(ds_2t.data_vars))
print("Coordinates:", list(ds_2t.coords))

DataNotFoundError: Cannot find data for variable='2t' at 2025-08-14T12:00:00.0.
Expected S3 URI: s3://met-office-atmospheric-model-data/global-deterministic/20250814T1200Z/20250814T1200Z_global-10km_2t.nc
Check that your ROOT_DIRECTORIES['MOGlobal10km'] is correct and that you have access to the bucket.

## 3. Global Map of 2-m Temperature

In [ ]:
# Extract the temperature field (convert K → °C for readability)
# xarray's .squeeze() removes any size-1 dimensions (e.g. single time step)
t2m = ds_2t["air_temperature"].squeeze() - 273.15  # K → °C

fig, ax = plt.subplots(
    figsize=(14, 7),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# Plot the data as a filled contour map
im = ax.contourf(
    t2m["longitude"],
    t2m["latitude"],
    t2m.values,
    levels=np.linspace(-50, 45, 40),
    cmap="RdBu_r",
    transform=ccrs.PlateCarree(),
)

# Add geographic features
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
ax.gridlines(draw_labels=True, linewidth=0.3, color="grey", alpha=0.5)

# Colourbar and labels
plt.colorbar(im, ax=ax, label="2-m Temperature (°C)", shrink=0.7, pad=0.02)
ax.set_title(
    f"Met Office Global 10 km – 2-m Temperature\n{QUERY_TIME} (UTC)",
    fontsize=14,
)

plt.tight_layout()
plt.savefig("global_10km_2t.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to global_10km_2t.png")

## 4. Load Multiple Variables

Load 10-metre wind components and compute wind speed.

In [ ]:
# Load both wind components in one call
accessor_wind = MOGlobal10km(["10u", "10v"], anon=True)
ds_wind = accessor_wind[QUERY_TIME]

# Compute wind speed
u = ds_wind["x_wind"].squeeze()
v = ds_wind["y_wind"].squeeze()
speed = np.sqrt(u**2 + v**2)

print("Wind speed statistics:")
print(f"  Min:  {float(speed.min()):.1f} m/s")
print(f"  Max:  {float(speed.max()):.1f} m/s")
print(f"  Mean: {float(speed.mean()):.1f} m/s")

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 7),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

im = ax.contourf(
    speed["longitude"],
    speed["latitude"],
    speed.values,
    levels=np.linspace(0, 30, 31),
    cmap="YlOrRd",
    transform=ccrs.PlateCarree(),
)

# Subsample for wind barbs to avoid over-plotting
step = max(1, len(speed.latitude) // 30)
lons = speed["longitude"].values[::step]
lats = speed["latitude"].values[::step]
U = u.values[::step, ::step]
V = v.values[::step, ::step]
ax.barbs(
    lons, lats, U, V,
    length=4, linewidth=0.5, color="k", alpha=0.5,
    transform=ccrs.PlateCarree(),
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
plt.colorbar(im, ax=ax, label="10-m Wind Speed (m/s)", shrink=0.7, pad=0.02)
ax.set_title(
    f"Met Office Global 10 km – 10-m Wind Speed\n{QUERY_TIME} (UTC)",
    fontsize=14,
)
plt.tight_layout()
plt.savefig("global_10km_wind.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to global_10km_wind.png")

## 5. Summary

In this notebook we:
1. Created a `MOGlobal10km` accessor with anonymous S3 access.
2. Loaded 2-m temperature and produced a global filled-contour map.
3. Loaded 10-m wind components, computed wind speed, and plotted a wind barb map.

See the companion script `scripts/demo_mo_global_10km.py` for the equivalent
command-line version.